# télos 25M Unified Upscaling Suite — Google Colab GPU Runner

This notebook executes the **25M Upscaled 3-Paradigm Suite** (AR, MDLM, UNDLM) on Google Colab / Cloud NVIDIA GPUs with fine-tuning learning rates (`max_lr = 1e-4`) to preserve pretrained token representations.

### Pipeline Overview
1. **Download Prerequisites**: Clones codebase, downloads tokenizers & 12.5M checkpoints directly from HuggingFace Hub.
2. **Upscaled Initialization**: Interpolates depth ($L=13 \to 8$) and pads hidden width ($d=256 \to 512$).
3. **3-Paradigm Execution**: Trains AR, MDLM, and UNDLM models across **1:1**, **1:10**, **1:20**, and **1:25** token ratios.
4. **Automated Export**: Evaluates 101 contextual probes and uploads trained 25M weights back to Hugging Face.

## 1. Environment Setup & GPU Diagnostic

In [ ]:
# Check GPU Availability
!nvidia-smi

# Install dependencies
!pip install -q torch torchvision safetensors huggingface_hub pyyaml tokenizers numpy matplotlib

## 2. Optional: Google Colab SSH Tunneling (Cloudflared / Tmate)
Run this cell if you want to SSH directly into the Colab instance from VSCode / Terminal.

In [ ]:
# Setup Tmate SSH server (one-command SSH access)
!apt-get install -y -qq tmate
!tmate -S /tmp/tmate.sock new-session -d
!tmate -S /tmp/tmate.sock wait tmate-ready
!tmate -S /tmp/tmate.sock display -p '#{tmate_ssh}'

## 3. Clone Repository & Download 12.5M Source Checkpoints from Hugging Face

In [ ]:
import os
from pathlib import Path
from huggingface_hub import snapshot_download, login

# Set your HF Repo and Token (optional if repo is public)
HF_REPO = "kazenoko/telos"
# login(token="YOUR_HF_TOKEN")

# Download tokenizer, configs, and 12.5M checkpoints
print(f"Downloading source weights and configs from https://huggingface.co/{HF_REPO}...")
snapshot_download(
    repo_id=HF_REPO,
    local_dir="./",
    allow_patterns=["configs/*", "checkpoints/ar/12m/*", "checkpoints/masked/12m/*", "checkpoints/uniform/12m/*", "scripts/*", "mdiff/*", "undiff/*", "ar/*"]
)
print("Download complete!")

## 4. Run 25M 1:1 Upscaled Suite (~25M Tokens, 191 Steps, `max_lr=1e-4`)

In [ ]:
!python scripts/colab/train_25m_upscaled.py --ratios r1 --hf-repo kazenoko/telos

## 5. Run 25M 1:10 Upscaled Suite (~250M Tokens, 1910 Steps, `max_lr=1e-4`)

In [ ]:
!python scripts/colab/train_25m_upscaled.py --ratios r10 --hf-repo kazenoko/telos

## 6. Run 25M 1:20 Upscaled Suite (~500M Tokens, 3820 Steps, `max_lr=1e-4`)

In [ ]:
!python scripts/colab/train_25m_upscaled.py --ratios r20 --hf-repo kazenoko/telos

## 7. Run 25M 1:25 Upscaled Suite (~625M Tokens, 4775 Steps, `max_lr=1e-4`)

In [ ]:
!python scripts/colab/train_25m_upscaled.py --ratios r25 --hf-repo kazenoko/telos

## 8. Upload Trained 25M Checkpoints to Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
print("Uploading 25M checkpoints to HuggingFace Hub...")
for paradigm in ["ar", "masked", "uniform"]:
    ckpt_25m_dir = f"checkpoints/{paradigm}/25m"
    if Path(ckpt_25m_dir).exists():
        api.upload_folder(
            folder_path=ckpt_25m_dir,
            path_in_repo=f"checkpoints/{paradigm}/25m",
            repo_id=HF_REPO,
            repo_type="model",
            allow_patterns=["*.safetensors", "*.json"]
        )
print("All 25M models uploaded successfully!")